# Type-B Training Pipeline

Supports two modes — set `MODE` in the Config cell:
- `'local'` : runs against local repo (VS Code + local kernel)
- `'colab'` : runs on Google Colab (mounts Drive, clones GitHub repo)

## Cell 1 — Config

### Experiment Matrix: CNN Model × Text Embedding (Type-B)

**Dataset**: 10,008 images · 128×128 RGB  
**Split** (seed=42): train 8,006 / val 1,001 / test 1,001 ← evaluated locally

#### CNN Models
| Key | Architecture | ~Params |
|-----|-------------|--------|
| `cnn_1layer` | 1 conv block → AvgPool(8×8) → FC(4096→512→dim) | 2.1 M |
| `cnn_3layer` | 3 conv blocks → AvgPool(2×2) → FC(512→512→dim) | 0.55 M |
| `resnet18_pt` | ResNet18 (ImageNet pretrained) → AdaptiveAvgPool(1×1) → FC(512→dim) | 11.4 M |

#### Text Embeddings
| Key | Method | Dim |
|-----|--------|-----|
| `sbert` | SBERT all-MiniLM-L6-v2 | 384 |
| `sbert_finetuned` | SBERT fine-tuned on Type-B pairs | 384 |
| `bert_mean` | BERT-base mean pool | 768 |
| `bert_pooler` | BERT-base \[CLS\] pooler | 768 |
| `tinybert_mean` | TinyBERT 4L-312D mean pool | 312 |
| `tinybert_pooler` | TinyBERT 4L-312D pooler | 312 |
| `word2vec_skipgram` | Skip-gram trained on Type-B corpus | 100 |
| `word2vec_pretrained` | Google News Word2Vec mean pool | 300 |
| `glove` | GloVe wiki-gigaword-300 mean pool | 300 |
| `tfidf_lsa` | TF-IDF + TruncatedSVD (LSA) | 100 |
| `tfidf_w2v` | TF-IDF weighted Word2Vec | 100 |

**→ Copy one config block from the reference cell below into the config cell, then run the notebook.**


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT CONFIG REFERENCE
# Copy ONE block's MODELS / EMBEDDINGS / RUN_TAG lines into the Config cell
# above, then run the notebook top-to-bottom.
# This cell is reference-only — running it does nothing.
#
# EXECUTION ORDER:
#   Stage 0 : B0      — baseline (cnn_1layer + tfidf_lsa)
#   Stage 1 : E2 a–k  — embedding sweep (cnn_1layer FIXED, vary embedding)
#   Stage 2 : E1 a,c,d — architecture sweep (best embedding from E2 FIXED, vary CNN)
#   Stage 3 : E3      — best CNN × best embedding
#   Stage 4 : LLM     — Gemini comparison (set RUN_GEMINI=True)
# ══════════════════════════════════════════════════════════════════════════════


# ── B0  BASELINE ──────────────────────────────────────────────────────────────
# cnn_1layer + tfidf_lsa: purely statistical embedding, simplest CNN.
# Establishes a near-zero retrieval floor (expected Top-1: 0–3%).
# The flat SVD spectrum of Type-B means number tokens are near-orthogonal
# and resist low-dimensional compression — confirmed structural limitation.
#
# MODELS     = ['cnn_1layer']
# EMBEDDINGS = ['tfidf_lsa']
# RUN_TAG    = 'b0_baseline'


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 1 — EMBEDDING SWEEP  (fixed model: cnn_1layer)
# Run E2 BEFORE E1. The best embedding from E2 becomes the fixed embedding
# for the architecture sweep (E1). Running E1 first would require assuming
# a best embedding a priori without experimental evidence.
# ══════════════════════════════════════════════════════════════════════════════

# E2j  TF-IDF + LSA  (100-dim)  — same as B0; E2 statistical floor anchor
# MODELS     = ['cnn_1layer']
# EMBEDDINGS = ['tfidf_lsa']
# RUN_TAG    = 'e2j_tfidf_lsa'

# E2k  TF-IDF weighted Word2Vec  (100-dim)
# MODELS     = ['cnn_1layer']
# EMBEDDINGS = ['tfidf_w2v']
# RUN_TAG    = 'e2k_tfidf_w2v'

# E2i  Skip-gram Word2Vec trained on Type-B corpus  (100-dim)
# MODELS     = ['cnn_1layer']
# EMBEDDINGS = ['word2vec_skipgram']
# RUN_TAG    = 'e2i_w2v_skip'

# E2g  GloVe wiki-gigaword-300, mean pool  (300-dim)
# MODELS     = ['cnn_1layer']
# EMBEDDINGS = ['glove']
# RUN_TAG    = 'e2g_glove'

# E2h  Google News Word2Vec mean pool  (300-dim)
# MODELS     = ['cnn_1layer']
# EMBEDDINGS = ['word2vec_pretrained']
# RUN_TAG    = 'e2h_w2v_pretrained'

# E2e  TinyBERT mean pool  (312-dim)
# MODELS     = ['cnn_1layer']
# EMBEDDINGS = ['tinybert_mean']
# RUN_TAG    = 'e2e_tinybert_mean'

# E2f  TinyBERT pooler  (312-dim)
# MODELS     = ['cnn_1layer']
# EMBEDDINGS = ['tinybert_pooler']
# RUN_TAG    = 'e2f_tinybert_pooler'

# E2a  SBERT pretrained  (384-dim)
# MODELS     = ['cnn_1layer']
# EMBEDDINGS = ['sbert']
# RUN_TAG    = 'e2a_sbert'

# E2b  SBERT fine-tuned on Type-B pairs  (384-dim)
# MODELS     = ['cnn_1layer']
# EMBEDDINGS = ['sbert_finetuned']
# RUN_TAG    = 'e2b_sbert_ft'

# E2c  BERT-base mean pool  (768-dim)  ← CANCELLED (out of scope)
# E2d  BERT-base [CLS] pooler  (768-dim)  ← CANCELLED (out of scope)


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 2 — ARCHITECTURE SWEEP  (fixed embedding: best from E2)
# Fill in EMBEDDINGS after reviewing E2 results.
# Expected best: sbert_finetuned — update if E2 shows otherwise.
# ══════════════════════════════════════════════════════════════════════════════

# E1a  1-layer CNN (same as E2 anchor — reuse E2a/E2b result, no re-run needed)
# MODELS     = ['cnn_1layer']
# EMBEDDINGS = ['<best_from_E2>']   # e.g. 'sbert_finetuned'
# RUN_TAG    = 'e1a_cnn1'

# E1c  3-layer CNN  (~0.55 M params)
# MODELS     = ['cnn_3layer']
# EMBEDDINGS = ['<best_from_E2>']
# RUN_TAG    = 'e1c_cnn3'

# E1d  ResNet18 pretrained (ImageNet)  (~11 M params)
# MODELS     = ['resnet18_pt']
# EMBEDDINGS = ['<best_from_E2>']
# RUN_TAG    = 'e1d_resnet_pt'


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 3 — BEST COMBINATION  (fill in after E1 + E2 results)
# ══════════════════════════════════════════════════════════════════════════════

# E3  best CNN × best embedding
# MODELS     = ['<best_from_E1>']       # e.g. 'resnet18_pt'
# EMBEDDINGS = ['<best_from_E2>']       # e.g. 'sbert_finetuned'
# RUN_TAG    = 'e3_best'


# ── FULL STAGE-1 SWEEP  (all E2 embeddings in one run — overnight Colab Pro) ──
# MODELS     = ['cnn_1layer']
# EMBEDDINGS = [
#     'tfidf_lsa', 'tfidf_w2v',
#     'word2vec_skipgram', 'word2vec_pretrained', 'glove',
#     'tinybert_mean', 'tinybert_pooler',
#     'sbert', 'sbert_finetuned',
# ]
# RUN_TAG    = 'e2_full_sweep'

print('Reference only — copy a block into the Config cell above.')

In [ ]:
# ── MODE ──────────────────────────────────────────────────────────────────────
MODE = 'colab'   # 'local' or 'colab'

# ── Local config (used when MODE='local') ─────────────────────────────────────
LOCAL_REPO_DIR = '/Users/yeon/work/UoB_CourseWork/uob-ds-intro-to-ai-final-cw-2026'

# ── Colab config (used when MODE='colab') ─────────────────────────────────────
# Token is loaded from Colab Secrets (left sidebar → key icon → add GITHUB_TOKEN)
# Never hardcode the token here — GitHub will auto-revoke it.
GITHUB_REPO_URL = 'https://github.com/Rylie-KIM/uob-ds-intro-to-ai-final-cw-2026'
DRIVE_BASE      = '/content/drive/MyDrive/uob-ds-ai'

# ── Experiment config ─────────────────────────────────────────────────────────
# Copy the relevant block from the reference cell above.
# Current: B0 baseline — cnn_1layer × tfidf_lsa
MODELS     = ['cnn_1layer']
EMBEDDINGS = ['tfidf_lsa']
EPOCHS     = 30
SKIP_EMBED = False
RUN_GEMINI = False

# ── Run tag — must match the reference block you copied ───────────────────────
RUN_TAG = 'b0_baseline'
# ──────────────────────────────────────────────────────────────────────────────

print(f'MODE = {MODE}  |  RUN_TAG = "{RUN_TAG}"')
print(f'MODELS = {MODELS}  |  EMBEDDINGS = {EMBEDDINGS}  |  EPOCHS = {EPOCHS}')

# Cell 2-1: Check corrupted images in the Colab's memeory content/images. 


In [ ]:
# Check corrupted images in the Colab's memeory content/images. 
from pathlib import Path                                                                                                
from PIL import Image                                                                                                   
                                                                                                                        
img_dir = Path('/content/images/type-b')                                                                                
corrupted = []                                                                                                          
                                                                                                                        
for f in sorted(img_dir.glob('*.png')):                                                                                 
    try:                                                                                                                
        Image.open(f).verify()                                                                                          
    except Exception:                                                                                                   
        corrupted.append(f)                                                                                             
                                                                                                                        
print(f'Corrupted: {len(corrupted)} files')                                                                             
for f in corrupted:                                                                                                     
    print(f'  deleting {f.name}')                                                                                       
    f.unlink()                                                                                                          
                                                                                                                        
print('Done. Re-run Cell 2 to re-copy missing files.')  

## Cell 2-2 — Setup (auto-selects local or Colab path)

In [ ]:
import os, sys, subprocess, shutil, time
from pathlib import Path

if MODE == 'local':
    REPO_DIR = LOCAL_REPO_DIR
    print(f'Local mode. Repo: {REPO_DIR}')

elif MODE == 'colab':
    from google.colab import drive, userdata
    drive.mount('/content/drive')

    REPO_DIR      = '/content/repo'
    DRIVE_DATA    = f'{DRIVE_BASE}/data'
    DRIVE_RESULTS = f'{DRIVE_BASE}/results'

    # Build authenticated clone URL from Colab Secrets
    try:
        token = userdata.get('GITHUB_TOKEN')
        repo_url_auth = GITHUB_REPO_URL.replace('https://', f'https://{token}@')
    except Exception:
        print('WARNING: GITHUB_TOKEN not found in Colab Secrets.')
        repo_url_auth = GITHUB_REPO_URL

    # Clone or pull repo
    if not os.path.exists(REPO_DIR):
        print('Cloning repo...')
        subprocess.run(['git', 'clone', repo_url_auth, REPO_DIR], check=True)
    else:
        print('Repo exists. Pulling latest...')
        # Reset local changes before pulling to avoid merge conflicts
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=True)
        commit = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD']).decode().strip()
        print(f'  HEAD: {commit}')

    # ── Copy train+val images to Colab local disk ────────────────────────────────
    # Test images are excluded: evaluation runs locally against the saved .pt
    # checkpoint, so Colab only needs train and val images.
    import pandas as _pd

    def _get_test_filenames(repo_dir: str) -> set:
        """Return the set of image filenames that belong to the test split."""
        splits_csv    = Path(repo_dir) / 'src' / 'data' / 'type-b' / 'splits' / 'type_b_splits_seed42.csv'
        image_map_csv = Path(repo_dir) / 'src' / 'data' / 'type-b' / 'image_map_b.csv'
        sentences_csv = Path(repo_dir) / 'src' / 'data' / 'type-b' / 'sentences_b.csv'
        splits_df  = _pd.read_csv(splits_csv)
        image_map  = _pd.read_csv(image_map_csv)
        sentences  = _pd.read_csv(sentences_csv)
        df         = image_map.merge(sentences, on='sentence_id')
        test_idx   = splits_df[splits_df['split'] == 'test']['idx'].tolist()
        return set(df.iloc[test_idx]['filename'].tolist())

    test_filenames = _get_test_filenames(REPO_DIR)
    print(f'[info] {len(test_filenames)} test images excluded (evaluated locally)')

    LOCAL_IMG_BASE = Path('/content/images')
    for dtype in ['type-b']:
        local_img = LOCAL_IMG_BASE / dtype
        drive_img = Path(DRIVE_DATA) / 'images' / dtype
        repo_img  = Path(REPO_DIR) / 'src' / 'data' / 'images' / dtype

        local_img.mkdir(parents=True, exist_ok=True)
        all_drive_files = sorted(drive_img.glob('*.png'))
        total           = len(all_drive_files)
        src_files   = [f for f in all_drive_files if f.name not in test_filenames]
        local_files = set(f.name for f in local_img.glob('*.png'))
        missing     = [f for f in src_files if f.name not in local_files]

        print(f'[info] images/{dtype}: {total} total | {len(src_files)} train+val | {total - len(src_files)} test (skipped)')
        if not missing:
            print(f'[skip] images/{dtype} — all {len(src_files)} train+val files already on local disk')
        else:
            print(f'Copying {len(missing)}/{len(src_files)} images ({dtype}) Drive → /content/images/{dtype}/  ({len(src_files) - len(missing)} already exist)')
            t0 = time.time()
            for i, src in enumerate(missing, 1):
                shutil.copy2(str(src), str(local_img / src.name))
                if i % 500 == 0 or i == len(missing):
                    elapsed = time.time() - t0
                    eta     = elapsed / i * (len(missing) - i)
                    print(f'  {i}/{len(missing)} ({i/len(missing)*100:.0f}%)  elapsed={elapsed:.0f}s  eta={eta:.0f}s', end='\r')
            print(f'\n[done] {len(missing)} images copied in {time.time()-t0:.1f}s')

        # Symlink repo → local disk (local→local: supported)
        if repo_img.is_symlink():
            os.unlink(str(repo_img))
        elif repo_img.exists():
            shutil.rmtree(str(repo_img))
        os.symlink(str(local_img), str(repo_img))
        print(f'[symlinked] repo/src/data/images/{dtype} → /content/images/{dtype}')

    # ── Restore results from Drive → local repo ───────────────────────────────
    for sub in ['checkpoints/type-b', 'metrics/type-b', 'figures/type-b']:
        drive_sub = Path(DRIVE_RESULTS) / sub
        repo_sub  = Path(REPO_DIR) / 'src' / 'pipelines' / 'results' / sub
        drive_sub.mkdir(parents=True, exist_ok=True)
        repo_sub.mkdir(parents=True, exist_ok=True)
        copied = 0
        for f in drive_sub.glob('*'):
            if f.is_file() and not (repo_sub / f.name).exists():
                shutil.copy2(str(f), str(repo_sub / f.name))
                copied += 1
        print(f'[restored] results/{sub}  ({copied} files from Drive)')

    # ── Copy embedding .pt from Drive ─────────────────────────────────────────
    drive_emb = Path(DRIVE_RESULTS) / 'embeddings'
    drive_emb.mkdir(parents=True, exist_ok=True)
    repo_emb = Path(REPO_DIR) / 'src' / 'embeddings' / 'computed-embeddings' / 'type-b' / 'results'
    repo_emb.mkdir(parents=True, exist_ok=True)
    for pt in drive_emb.glob('*.pt'):
        dest = repo_emb / pt.name
        if not dest.exists():
            shutil.copy2(str(pt), str(dest))
            print(f'[copied] {pt.name}')

    print(f'\nColab mode ready. Repo: {REPO_DIR}')

# Add repo to sys.path
for p in [REPO_DIR, str(Path(REPO_DIR) / 'src'), str(Path(REPO_DIR) / 'src' / 'embeddings' / 'non-pretrained')]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

## Cell 3 — Install Dependencies

In [ ]:
if MODE == 'colab':
    !pip install -q sentence-transformers gensim python-docx datasets
    print('Dependencies installed.')
else:
    print('Local mode — using existing venv. Skipping pip install.')

## Cell 4 — Verify Paths

In [ ]:
checks = {
    'sentences_b.csv' : Path(REPO_DIR) / 'src' / 'data' / 'type-b' / 'sentences_b.csv',
    'image_map_b.csv' : Path(REPO_DIR) / 'src' / 'data' / 'type-b' / 'image_map_b.csv',
    'images/type-b/'  : Path(REPO_DIR) / 'src' / 'data' / 'images' / 'type-b',
    'splits CSV'      : Path(REPO_DIR) / 'src' / 'data' / 'type-b' / 'splits',
    'train_type_b.py' : Path(REPO_DIR) / 'src' / 'pipelines' / 'training' / 'type-b' / 'train_type_b.py',
    'generate_embed'  : Path(REPO_DIR) / 'src' / 'embeddings' / 'computed-embeddings' / 'type-b' / 'generate_embeddings_type_b.py',
}
all_ok = True
for name, path in checks.items():
    ok = path.exists()
    print(f'  {"OK" if ok else "MISSING"}  {name:25s}  {path}')
    if not ok:
        all_ok = False
print()
print('All paths OK — ready to run.' if all_ok else 'Fix missing paths before continuing.')

## Cell 5 — Generate Embeddings

In [ ]:
generate_script  = Path(REPO_DIR) / 'src' / 'embeddings' / 'computed-embeddings' / 'type-b' / 'generate_embeddings_type_b.py'
embed_results_dir = Path(REPO_DIR) / 'src' / 'embeddings' / 'computed-embeddings' / 'type-b' / 'results'

if SKIP_EMBED:
    print('Skipping embedding generation (SKIP_EMBED=True)')
else:
    for emb in EMBEDDINGS:
        pt_path = embed_results_dir / f'{emb}_embedding_result_typeb.pt'
        if pt_path.exists():
            print(f'[skip] {emb} — already exists')
            continue
        print(f'[generating] {emb}...')
        subprocess.run(
            [sys.executable, str(generate_script), '--embedding', emb],
            cwd=REPO_DIR,
            check=True,
        )
        print(f'[done] {emb}')

## Cell 6 — Train

In [ ]:
import importlib
import importlib.util as _ilu
import sys

# Force-evict any stale cached version of src.config.paths from sys.modules.
# git reset --hard updates files on disk but does NOT clear sys.modules, so
# a previously imported (old) paths.py can survive across cell re-runs and
# cause run_experiment to save to the wrong directory.
for _key in list(sys.modules.keys()):
    if 'config.paths' in _key or _key in ('src.config', 'config'):
        del sys.modules[_key]

_spec = _ilu.spec_from_file_location(
    'train_type_b',
    Path(REPO_DIR) / 'src' / 'pipelines' / 'training' / 'type-b' / 'train_type_b.py',
)
_mod = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

# Confirm paths the loaded module will use
print(f'[paths] METRICS_B     = {_mod.METRICS_B}')
print(f'[paths] CHECKPOINTS_B = {_mod.CHECKPOINTS_B}')
assert str(_mod.METRICS_B).endswith('metrics/type-b'), \
    f'METRICS_B path wrong: {_mod.METRICS_B}'
assert str(_mod.CHECKPOINTS_B).endswith('checkpoints/type-b'), \
    f'CHECKPOINTS_B path wrong: {_mod.CHECKPOINTS_B}'
print('[paths] OK\n')


def _backup_results_to_drive():
    """Copy all result files from local repo → Drive immediately after training."""
    if MODE != 'colab':
        return
    for sub in ['checkpoints/type-b', 'metrics/type-b', 'figures/type-b']:
        repo_sub  = Path(REPO_DIR) / 'src' / 'pipelines' / 'results' / sub
        drive_sub = Path(DRIVE_RESULTS) / sub
        drive_sub.mkdir(parents=True, exist_ok=True)
        print(f'  [backup] {sub}')
        print(f'    src : {repo_sub}')
        print(f'    dst : {drive_sub}')
        copied = 0
        for f in sorted(repo_sub.glob('*')):
            if f.is_file():
                shutil.copy2(str(f), str(drive_sub / f.name))
                print(f'    + {f.name}')
                copied += 1
        if copied == 0:
            print(f'    (no files found in src)')


total = len(MODELS) * len(EMBEDDINGS)
for i, model_name in enumerate(MODELS):
    for j, emb_name in enumerate(EMBEDDINGS):
        n = i * len(EMBEDDINGS) + j + 1
        print(f'\n[{n}/{total}] {model_name} × {emb_name}  tag={RUN_TAG!r}')
        _mod.run_experiment(
            model_name=model_name,
            embedding_name=emb_name,
            epochs=EPOCHS,
            device=device,
            run_tag=RUN_TAG,
        )
        # Back up immediately after each run so results survive a Colab disconnect
        _backup_results_to_drive()

print('\nAll done.')

In [ ]:
## Cell 6.5 — (Optional) Manual re-backup to Drive
# Cell 6 already backs up after each run. Re-run this only if you need
# to force-sync (e.g. after editing a file directly in the repo).
if MODE == 'colab':
    for sub in ['checkpoints/type-b', 'metrics/type-b', 'figures/type-b']:
        repo_sub  = Path(REPO_DIR) / 'src' / 'pipelines' / 'results' / sub
        drive_sub = Path(DRIVE_RESULTS) / sub
        drive_sub.mkdir(parents=True, exist_ok=True)
        copied = 0
        for f in repo_sub.glob('*'):
            if f.is_file():
                shutil.copy2(str(f), str(drive_sub / f.name))
                copied += 1
        print(f'[backed up] results/{sub} → Drive  ({copied} files)')
else:
    print('Local mode — Drive backup skipped.')

## Cell 7 — Results & Loss Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics_dir = Path(REPO_DIR) / 'src' / 'pipelines' / 'results' / 'metrics' / 'type-b'
figures_dir = Path(REPO_DIR) / 'src' / 'pipelines' / 'results' / 'figures' / 'type-b'
figures_dir.mkdir(parents=True, exist_ok=True)

for model_name in MODELS:
    for emb_name in EMBEDDINGS:
        run_name = f'b_{model_name}_{emb_name}' + (f'_{RUN_TAG}' if RUN_TAG else '')
        print(f'\n{"="*60}')
        print(f'  {run_name}')
        print(f'{"="*60}')

        # ── epoch_log.csv ──────────────────────────────────────────
        epoch_log_path = metrics_dir / f'{run_name}_epoch_log.csv'
        if epoch_log_path.exists():
            log = pd.read_csv(epoch_log_path)
            print(f'\n[epoch_log]  {epoch_log_path.name}  ({len(log)} epochs)')
            display(log)
        else:
            print(f'[missing] {epoch_log_path.name}')

        # ── predictions.csv ────────────────────────────────────────
        pred_path = metrics_dir / f'{run_name}_predictions.csv'
        if pred_path.exists():
            pred = pd.read_csv(pred_path)
            print(f'\n[predictions]  {pred_path.name}  ({len(pred)} samples)')
            display(pred.head(10))
        else:
            print(f'[missing] {pred_path.name}')

        # ── Loss curves ────────────────────────────────────────────
        if epoch_log_path.exists():
            fig, axes = plt.subplots(1, 2, figsize=(14, 4))
            fig.suptitle(f'Loss Curves — {run_name}')
            axes[0].plot(log['epoch'], log['train_loss'], label='train')
            axes[1].plot(log['epoch'], log['val_loss'],   label='val', color='orange')
            for ax, title in zip(axes, ['Train Loss', 'Val Loss']):
                ax.set_title(title); ax.set_xlabel('Epoch')
                ax.legend(); ax.grid(True, alpha=0.3)
            out = figures_dir / f'loss_curves_{run_name}.png'
            fig.savefig(out, dpi=150, bbox_inches='tight')
            plt.show()
            print(f'[saved] {out.name}')

# ── Per-run summaries ─────────────────────────────────────────────────────
for model_name in MODELS:
    for emb_name in EMBEDDINGS:
        run_name = f'b_{model_name}_{emb_name}' + (f'_{RUN_TAG}' if RUN_TAG else '')
        run_summary_path = metrics_dir / f'{run_name}_summary.csv'
        print(f'\n{"="*60}')
        print(f'  {run_name}_summary.csv')
        print(f'{"="*60}')
        if run_summary_path.exists():
            display(pd.read_csv(run_summary_path))
        else:
            print(f'[missing] {run_summary_path.name}')

# ── Aggregate summary (all runs) ───────────────────────────────────────────
print(f'\n{"="*60}')
print('  results_summary.csv  (all runs)')
print(f'{"="*60}')
agg_path = metrics_dir / 'results_summary.csv'
if agg_path.exists():
    display(pd.read_csv(agg_path).sort_values('top_1_acc', ascending=False))
else:
    print('[missing] results_summary.csv')

# ── Back up figures to Drive now that they have been generated ─────────────
if MODE == 'colab':
    drive_fig = Path(DRIVE_RESULTS) / 'figures' / 'type-b'
    drive_fig.mkdir(parents=True, exist_ok=True)
    copied = 0
    for f in figures_dir.glob('*.png'):
        shutil.copy2(str(f), str(drive_fig / f.name))
        copied += 1
    print(f'\n[backed up] results/figures/type-b → Drive  ({copied} files)')

## Cell 8 — Push Results to GitHub

In [ ]:
tag_suffix = f' tag={RUN_TAG}' if RUN_TAG else ''
COMMIT_MSG = f'Add results: {MODELS} x {EMBEDDINGS}{tag_suffix}'
_push_script = str(Path(REPO_DIR) / 'notebooks' / 'train-evaluation' / 'type-b' / '_push_results.py')
%run -i $_push_script

## Cell 9 — (Optional) Gemini Comparison

In [ ]:
if RUN_GEMINI:
    # os.environ['GEMINI_API_KEY'] = 'your-key-here'
    gemini_script = Path(REPO_DIR) / 'src' / 'pipelines' / 'evaluation' / 'type-b' /'gemini_comparison.py'
    subprocess.run([sys.executable, str(gemini_script)], cwd=REPO_DIR, check=True)
else:
    print('Skipping Gemini (RUN_GEMINI=False)')